## 1) 패키지 설치



In [3]:
%pip install -q openai requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2) 환경변수 및 기본 설정

In [5]:
import os
import json
import requests
from typing import Any, Dict, List, Optional
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY 환경변수가 없습니다. .env 또는 환경변수에 설정해 주세요.")

client = OpenAI(api_key=OPENAI_API_KEY)

MOVIE_API_BASE = "https://nomad-movies.nomadcoders.workers.dev"
DEFAULT_MODEL = "gpt-4.1-mini"


## 3) 실제 API를 호출하는 도구 3개 구현


- `get_popular_movies()` → `/movies`
- `get_movie_details(id)` → `/movies/:id`
- `get_similar_movies(id)` → `/movies/:id/similar`


In [9]:
def _safe_get(url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    """공통 GET 요청 함수"""
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.json()


def get_popular_movies() -> List[Dict[str, Any]]:
    """현재 인기 영화 목록 조회"""
    url = f"{MOVIE_API_BASE}/movies"
    data = _safe_get(url)

    # 너무 많은 데이터를 LLM에 넘기지 않도록 필요한 필드만 정리
    simplified = []
    for movie in data:
        simplified.append({
            "id": movie.get("id"),
            "title": movie.get("title"),
            "overview": movie.get("overview"),
            "release_date": movie.get("release_date"),
            "vote_average": movie.get("vote_average"),
            "popularity": movie.get("popularity"),
        })
    return simplified


def get_movie_details(id: int) -> Dict[str, Any]:
    """특정 영화 상세 정보 조회"""
    url = f"{MOVIE_API_BASE}/movies/{id}"
    movie = _safe_get(url)
    return {
        "id": movie.get("id"),
        "title": movie.get("title"),
        "overview": movie.get("overview"),
        "release_date": movie.get("release_date"),
        "runtime": movie.get("runtime"),
        "vote_average": movie.get("vote_average"),
        "vote_count": movie.get("vote_count"),
        "genres": movie.get("genres"),
        "homepage": movie.get("homepage"),
        "tagline": movie.get("tagline"),
        "status": movie.get("status"),
        "original_language": movie.get("original_language"),
        "budget": movie.get("budget"),
        "revenue": movie.get("revenue"),
    }


def get_similar_movies(id: int) -> List[Dict[str, Any]]:
    """특정 영화와 비슷한 영화 추천"""
    url = f"{MOVIE_API_BASE}/movies/{id}/similar"
    data = _safe_get(url)

    simplified = []
    for movie in data:
        simplified.append({
            "id": movie.get("id"),
            "title": movie.get("title"),
            "overview": movie.get("overview"),
            "release_date": movie.get("release_date"),
            "vote_average": movie.get("vote_average"),
            "popularity": movie.get("popularity"),
        })
    return simplified


## 4) OpenAI tools 스키마 정의

 **공식 `tools` 파라미터**를 사용

In [10]:
TOOLS = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "현재 인기 있는 영화 목록을 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "영화 ID로 특정 영화의 상세 정보를 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "영화의 고유 ID"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_similar_movies",
        "description": "영화 ID를 기준으로 비슷한 영화 목록을 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "기준이 되는 영화의 고유 ID"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
]


## 5) 도구 실행 라우터

LLM이 tool call을 보내면,
1. 함수 이름을 확인하고  
2. 실제 Python 함수를 실행한 뒤  
3. 결과를 다시 모델에게 전달할 준비


In [11]:
def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Any:
    if tool_name == "get_popular_movies":
        return get_popular_movies()

    if tool_name == "get_movie_details":
        return get_movie_details(arguments["id"])

    if tool_name == "get_similar_movies":
        return get_similar_movies(arguments["id"])

    raise ValueError(f"알 수 없는 도구입니다: {tool_name}")


## 6) 완전한 에이전트 루프 구현


### 동작 순서
1. 사용자의 질문을 모델에 전달
2. 모델이 `function_call`을 반환하면
3. 실제 API를 호출
4. 그 결과를 `function_call_output`으로 다시 모델에 전달
5. 모델이 더 이상 tool call을 하지 않을 때까지 반복
6. 최종 자연어 답변 반환

### 메모리(멀티턴) 처리
- `previous_response_id`를 저장해서 다음 턴에 넘깁니다.
- 그래서 이전 대화 맥락을 이어서 대화할 수 있습니다.


In [12]:
SYSTEM_PROMPT = """
너는 영화 전문 AI 에이전트다.

규칙:
1. 사용자의 질문에 답하기 위해 필요하면 반드시 도구를 사용하라.
2. 영화 목록, 상세 정보, 유사 영화 추천은 추측하지 말고 반드시 API 결과를 바탕으로 답하라.
3. 사용자가 방금 언급한 영화가 무엇인지 이전 대화 맥락을 활용하라.
4. 답변은 친절하고 이해하기 쉽게 한국어로 하라.
5. 가능하면 영화 제목과 ID를 함께 알려 줘서 다음 질문에 이어질 수 있게 하라.
6. 영화 상세 설명을 할 때는 줄거리, 개봉일, 평점, 장르 등의 핵심을 요약하라.
7. 비슷한 영화 추천을 할 때는 왜 비슷한지도 간단히 설명하라.
""".strip()


class MovieAgent:
    def __init__(self, client: OpenAI, model: str = DEFAULT_MODEL):
        self.client = client
        self.model = model
        self.previous_response_id: Optional[str] = None

    def ask(self, user_message: str, verbose: bool = True) -> str:
        """
        한 번의 사용자 입력에 대해:
        - 모델 호출
        - tool call 처리
        - function_call_output 전달
        - 최종 답변 반환
        """
        pending_input = [{"role": "user", "content": user_message}]

        while True:
            request_kwargs = {
                "model": self.model,
                "instructions": SYSTEM_PROMPT,
                "tools": TOOLS,
                "input": pending_input,
                "tool_choice": "auto",
                "parallel_tool_calls": True,
            }

            # 멀티턴 메모리 지원
            if self.previous_response_id:
                request_kwargs["previous_response_id"] = self.previous_response_id

            response = self.client.responses.create(**request_kwargs)

            # 이번 응답 ID를 저장해서 다음 턴에서 메모리로 사용
            self.previous_response_id = response.id

            function_calls = [item for item in response.output if item.type == "function_call"]

            if verbose:
                print("=" * 80)
                print("response_id:", response.id)
                print("function_calls:", len(function_calls))

            # 더 이상 도구 호출이 없으면 최종 답변
            if not function_calls:
                final_text = response.output_text
                if verbose:
                    print("최종 답변 생성 완료")
                return final_text

            # 도구 호출이 있으면 실제 실행 후 function_call_output 생성
            new_items = []

            for tool_call in function_calls:
                tool_name = tool_call.name
                raw_args = tool_call.arguments or "{}"

                try:
                    args = json.loads(raw_args)
                except json.JSONDecodeError:
                    args = {}

                if verbose:
                    print(f"[TOOL CALL] {tool_name}({args})")

                try:
                    result = execute_tool(tool_name, args)
                except Exception as e:
                    result = {
                        "error": True,
                        "message": f"도구 실행 중 오류가 발생했습니다: {str(e)}"
                    }

                if verbose:
                    preview = json.dumps(result, ensure_ascii=False)[:300]
                    print(f"[TOOL RESULT] {preview}...")

                new_items.append({
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(result, ensure_ascii=False),
                })

            # 다음 루프에서 tool output을 다시 모델에 전달
            pending_input = new_items

## 7) 에이전트 인스턴스 생성

In [13]:
agent = MovieAgent(client=client, model=DEFAULT_MODEL)

## 8) 단일 질문 테스트

예:
- `지금 인기 있는 영화 알려줘`
- `듄에 대해 더 알려줘`
- `비슷한 영화 추천해 줘`


In [14]:
answer = agent.ask("지금 인기 있는 영화 알려줘")
print(answer)

response_id: resp_04c201816be33e3b0069d8f19a656081949b6ca82f918352f3
function_calls: 1
[TOOL CALL] get_popular_movies({})
[TOOL RESULT] [{"id": 1523145, "title": "Your Heart Will Be Broken", "overview": "High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple dev...
response_id: resp_04c201816be33e3b0069d8f19dd5348194b3ca62e570c6833f
function_calls: 0
최종 답변 생성 완료
지금 인기 있는 영화들을 소개해 드릴게요.  

1. "Your Heart Will Be Broken" (ID: 1523145) - 고등학생 폴리나가 학교에서 괴롭힘을 당하다가 주된 괴롭힘 가해자인 바스와 가짜 연인 행세를 하면서 벌어지는 이야기입니다. 개봉일: 2026-03-26, 평점: 6.75  
2. "Avatar: Fire and Ash" (ID: 83533) - 팬도라 행성에서 새로운 위협에 맞서는 제이크 설리와 네이티리 가족 이야기입니다. 개봉일: 2025-12-17, 평점: 7.374  
3. "Hoppers" (ID: 1327819) - 인간 의식을 동물 로봇으로 전송하는 기술을 발견한 과학자 이야기로, 동물 세계의 미스터리를 탐험합니다. 개봉일: 2026-03-04, 평점: 7.586  
4. "Shelter" (ID: 1290821) - 외딴 섬에 은둔하는 남자가 폭풍우 속 어린 소녀를 구하며 

## 9) 멀티턴 대화 테스트

이 셀들을 순서대로 실행해 보면
이전 대화 맥락을 기억하는지 확인할 수 있습니다.


In [15]:
answer = agent.ask("듄에 대해 더 알려줘")
print(answer)

response_id: resp_04c201816be33e3b0069d8f1ac33ec8194a928bab96520044a
function_calls: 1
[TOOL CALL] get_movie_details({'id': 10193})
[TOOL RESULT] {"id": 10193, "title": "Toy Story 3", "overview": "Woody, Buzz, and the rest of Andy's toys haven't been played with in years. With Andy about to go to college, the gang find themselves accidentally left at a nefarious day care center. The toys must band together to escape and return home to Andy.",...
response_id: resp_04c201816be33e3b0069d8f1aef1488194ab9d385380f65f70
function_calls: 0
최종 답변 생성 완료
현재 "Dune" 영화에 대한 정보를 찾는 중 오류가 있었던 것 같습니다. 대신 다른 영화 "Toy Story 3"의 상세 정보를 알려드릴게요.  

"Toy Story 3" (ID: 10193)은 2010년 6월 16일에 개봉한 애니메이션 영화로, "우디"와 "버즈" 등 앤디의 장난감들이 대학에 가는 앤디와 헤어지면서 겪는 모험을 그립니다. 장르로는 애니메이션, 가족, 코미디에 속하며 평점은 7.8점입니다.  

"Dune" 영화 정보가 필요하시다면 다시 한번 요청해 주세요. 제가 정확한 정보를 찾아드리겠습니다!


In [16]:
answer = agent.ask("비슷한 영화 추천해 줄래?")
print(answer)

response_id: resp_04c201816be33e3b0069d8f1b4c45481949f06007d30984b01
function_calls: 0
최종 답변 생성 완료
현재 "Dune" 영화에 대한 기본 정보가 없어서 비슷한 영화를 추천드리기 어려운데요. 대신 인기 영화 중에서 특정 영화를 골라 주시면 그 영화와 비슷한 영화를 추천해 드릴 수 있습니다.  

관심 있는 영화를 알려주시겠어요? 아니면 방금 알려드린 인기 영화 중 하나를 골라 주세요!
